# 57 — M6 Gate

Gate computable de Productionization & MLOps. El cierre de M6 valida scoring, lineage, monitoring, audit, rollback y human review. El deployment status se reporta por separado y puede ser `SHADOW_READY` o `OPERATIONALLY_READY_PROMOTION_BLOCKED`; la promoción autónoma a producción permanece deshabilitada.

> **Closure reconciliation note (2026-09-23).** The cached output that previously showed `M6 ❌ OPEN` was generated before the final successful reruns of component evidence. It has been removed rather than rewritten. The reconciled closure decision and provenance are documented in `docs/decisions/M6_GATE.md` and `docs/decisions/M6_CLOSURE_REVIEW_2026-09-23.md`. Re-executing this notebook in the project environment will regenerate the canonical Parquet gate report from current component reports.


In [ ]:
from pathlib import Path

from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec
from argos.config.modeling import load_modeling_spec
from argos.config.production import load_production_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

m5_gate = paths.docs / "decisions" / "M5_GATE.md"
assert m5_gate.exists() and "M5 ✅ CLOSED" in m5_gate.read_text(encoding="utf-8"), (
    "M5 no está formalmente cerrado. No iniciar M6."
)

GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")
MODEL_SPEC = load_modeling_spec(paths.configs / "modeling.toml")
M6_SPEC = load_production_spec(paths.configs / "production.toml")
MANIFEST_DIR = paths.root / "artifacts" / "manifests"
TEMP_DIR = paths.root / ".cache" / "duckdb"
SHADOW_INPUT = paths.root / "data" / "m5" / "fresh_oot_2026.parquet"

assert GOLD_SPEC.schema_version == M6_SPEC.core.benchmark_gold_schema
assert MODEL_SPEC.schema_version == M6_SPEC.core.benchmark_model_schema

{
    "m6_schema": M6_SPEC.schema_version,
    "model_version": M6_SPEC.core.model_version,
    "operating_mode": M6_SPEC.core.operating_mode,
    "scoring_mode": M6_SPEC.core.scoring_mode,
    "human_review_required": M6_SPEC.core.human_review_required,
    "production_promotion_authorized": M6_SPEC.deployment.allow_production_promotion,
}


In [ ]:
from argos.evaluation.m6_gate import evaluate_m6_gate

decision = evaluate_m6_gate(
    root=paths.root,
    production_spec=M6_SPEC,
    reports_root=paths.reports,
    manifest_dir=MANIFEST_DIR,
    docs_root=paths.docs,
)
decision.report

In [ ]:
blocking = decision.report.loc[decision.report["blocking"]]
print(f"M6 ✅ CLOSED — {decision.deployment_status}" if decision.closed else "M6 ❌ OPEN")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Deployment status:", decision.deployment_status)
print("Decision artifact:", decision.decision_artifact)
assert decision.closed